# 🧠 Step 5: Gen AI — Investigation Summary Generator

This notebook uses **Databricks Foundation Model APIs** to generate
natural-language investigation summaries for high-severity alerts.

For each critical/high alert, the AI generates:
- A concise summary of what happened
- Risk assessment
- Recommended next steps for the analyst

In [0]:
%run ./01_config

# ⚙️ Configuration — SOC Alert Triage

Shared configuration for all notebooks. Update the catalog name below before running.

✅ Configuration loaded
   Catalog: rajarora_febar
   Schema:  soc_alert_triage
   Tables:  10 tables configured


In [0]:
from pyspark.sql import functions as F

# Get high-severity alerts that need investigation summaries
high_severity_alerts = spark.sql(f"""
    SELECT 
        e.event_id,
        e.event_timestamp,
        e.source_type,
        e.customer_name,
        e.industry,
        e.severity_score,
        e.composite_risk_score,
        e.risk_tier,
        e.threat_intel_match,
        e.threat_category,
        e.action,
        e.src_ip,
        e.dst_ip,
        e.username,
        e.alert_type,
        e.mitre_tactic,
        e.process_name,
        e.asset_type,
        e.criticality,
        e.department,
        p.ml_predicted_severity,
        p.ml_confidence
    FROM {TABLES['enriched_alerts']} e
    LEFT JOIN {TABLES['alert_predictions']} p ON e.event_id = p.event_id
    WHERE e.risk_tier IN ('critical', 'high')
    LIMIT 500
""")

alert_count = high_severity_alerts.count()
print(f"📊 Found {alert_count:,} high-severity alerts for investigation summary generation")

📊 Found 500 high-severity alerts for investigation summary generation


In [0]:
# Use Databricks AI Functions to generate investigation summaries
# ai_query calls Foundation Model APIs (DBRX, Llama, etc.)

investigation_summaries = spark.sql(f"""
    SELECT 
        event_id,
        event_timestamp,
        source_type,
        customer_name,
        risk_tier,
        composite_risk_score,
        ai_query(
            'databricks-meta-llama-3-3-70b-instruct',
            CONCAT(
                'You are a senior SOC analyst. Generate a concise investigation summary for this security alert. ',
                'Include: (1) What happened, (2) Risk assessment, (3) Recommended next steps. Keep it under 150 words.\\n\\n',
                'Alert Details:\\n',
                '- Source: ', source_type, '\\n',
                '- Customer: ', customer_name, ' (', COALESCE(industry, 'Unknown'), ')\\n',
                '- Severity: ', CAST(severity_score AS STRING), '/10\\n',
                '- Risk Tier: ', risk_tier, '\\n',
                '- Threat Intel Match: ', CAST(threat_intel_match AS STRING), '\\n',
                CASE WHEN threat_category IS NOT NULL THEN CONCAT('- Threat Category: ', threat_category, '\\n') ELSE '' END,
                CASE WHEN alert_type IS NOT NULL THEN CONCAT('- Alert Type: ', alert_type, '\\n') ELSE '' END,
                CASE WHEN mitre_tactic IS NOT NULL THEN CONCAT('- MITRE Tactic: ', mitre_tactic, '\\n') ELSE '' END,
                CASE WHEN src_ip IS NOT NULL THEN CONCAT('- Source IP: ', src_ip, '\\n') ELSE '' END,
                CASE WHEN username IS NOT NULL THEN CONCAT('- User: ', username, '\\n') ELSE '' END,
                CASE WHEN process_name IS NOT NULL THEN CONCAT('- Process: ', process_name, '\\n') ELSE '' END,
                '- Asset Criticality: ', COALESCE(criticality, 'unknown'), '\\n',
                '- ML Predicted Severity: ', COALESCE(ml_predicted_severity, 'N/A'), 
                ' (confidence: ', COALESCE(CAST(ml_confidence AS STRING), 'N/A'), ')'
            )
        ) AS investigation_summary,
        current_timestamp() AS summary_generated_at
    FROM (
        SELECT 
            e.event_id, e.event_timestamp, e.source_type, e.customer_name,
            e.industry, e.severity_score, e.composite_risk_score, e.risk_tier,
            e.threat_intel_match, e.threat_category, e.action, e.src_ip,
            e.username, e.alert_type, e.mitre_tactic, e.process_name,
            e.criticality,
            p.ml_predicted_severity, p.ml_confidence
        FROM {TABLES['enriched_alerts']} e
        LEFT JOIN {TABLES['alert_predictions']} p ON e.event_id = p.event_id
        WHERE e.risk_tier IN ('critical', 'high')
        LIMIT 100
    )
""")

# Save investigation summaries
investigation_summaries.write.mode("overwrite").saveAsTable(TABLES["investigation_summaries"])
summary_count = spark.table(TABLES["investigation_summaries"]).count()
print(f"✅ Generated {summary_count:,} AI investigation summaries")

✅ Generated 100 AI investigation summaries


In [0]:
spark.sql(f"""
    SELECT 
        event_id,
        source_type,
        customer_name,
        risk_tier,
        SUBSTRING(investigation_summary, 1, 500) as summary_preview
    FROM {TABLES['investigation_summaries']}
    LIMIT 5
""").display()

event_id,source_type,customer_name,risk_tier,summary_preview
e840fb6b-5fbb-485e-a159-ea34e581c791,email,Global Technologies,high,"**Investigation Summary** A high-severity security alert was triggered by an email source, affecting Global Technologies (Retail). The alert has a high risk tier and asset criticality, with a machine learning predicted severity of critical (confidence: 0.9649). **Risk Assessment**: The high severity and criticality of the alert pose a significant risk to Global Technologies' retail operations. **Recommended Next Steps**: Immediately investigate the email source, contain potential threats, an"
8c73708f-d775-405c-bee6-4568b17e71cd,email,Acme Technologies,high,"**Investigation Summary** A high-severity security alert was triggered by an email sent to Acme Technologies, a critical energy sector customer. The alert has a risk tier of high and asset criticality of critical, with a machine learning predicted severity of critical (confidence: 0.7078). **Risk Assessment**: The risk is considered high due to the customer's critical infrastructure and potential impact on the energy sector. **Recommended Next Steps**: Immediately investigate the email conte"
de1b76c3-c250-43b2-8c45-bb7e6e591af5,cloud_audit,Acme LLC,high,"**Investigation Summary** (1) A high-severity alert was triggered by cloud_audit for Acme LLC (Retail), indicating potential unauthorized activity by IAM user admin_6, accessing a critical asset. (2) Risk assessment: High risk due to the criticality of the asset and predicted severity, potentially leading to data breaches or disruption of business operations. (3) Recommended next steps: Immediately investigate admin_6's activity, review access logs, and verify user authenticity. Consider revoki"
4853ced3-4f4c-4ad7-8ee2-ff9ca5a38501,firewall,Pacific Systems,high,"**Investigation Summary** A high-severity alert was triggered by the firewall at Pacific Systems (Energy), indicating potential malicious activity from source IP 86.245.83.249. The alert has a severity score of 9/10 and is classified as high-risk, with a critical asset impacted. **Risk Assessment**: High risk due to the criticality of the asset and high predicted severity. **Recommended Next Steps**: Immediately block the source IP, initiate a thorough investigation, and perform a detailed an"
dd1d7d38-8367-4d55-9d7d-6ab7e0212f05,firewall,Nexus Systems,high,"**Investigation Summary** A high-severity alert was triggered by the firewall at Nexus Systems (Education), indicating potential malicious activity from source IP 23.208.14.164. The asset targeted is critical, and machine learning models predict high severity with moderate confidence (0.5061). **Risk Assessment**: High risk due to the criticality of the asset and potential impact on the education system. **Recommended Next Steps**: Immediately block the source IP, conduct a thorough investiga"


## ✅ Gen AI Investigation Summaries Complete!

### What was built:
- **Foundation Model API** (Llama 3.3 70B) generates investigation summaries
- Each summary includes: what happened, risk assessment, recommended next steps
- Summaries are stored in `gold_investigation_summaries` table
- Integrated with ML predictions for richer context

### Business Value:
- Saves analysts **10-15 minutes per alert** on initial investigation
- Provides consistent, structured investigation starting points
- Combines threat intel, ML predictions, and contextual data

**Next step**: Run `06_lakebase` to set up operational serving